In [7]:
import pandas as pd
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import  StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

#load and clean titanic dataset
df = sns.load_dataset('titanic')
df = df[['survived', 'pclass', 'embarked', 'age', 'fare', 'sex']].dropna()
df['sex'] = df['sex'].map({'male': 0, 'female': 1})
df['embarked'] = df['embarked'].map({'S': 0, 'C': 1, 'Q': 2}) 

X = df.drop('survived', axis = 1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

#Build pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators = 100, random_state = 42))
])

#train: 
pipe.fit(X_train, y_train)

#Evaluate
print(f" Train: {pipe.score(X_train, y_train):.4f}")
print(f" Test: {pipe.score(X_test, y_test):.4f}")
print(f"Classification Report: {classification_report(y_test, pipe.predict(X_test))}")
     

 Train: 0.9859
 Test: 0.7762
Classification Report:               precision    recall  f1-score   support

           0       0.78      0.84      0.81        80
           1       0.77      0.70      0.73        63

    accuracy                           0.78       143
   macro avg       0.78      0.77      0.77       143
weighted avg       0.78      0.78      0.77       143



In [9]:
from sklearn.model_selection import GridSearchCV
#Define parameter: (format: stepname__parameter)

param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__max_depth': [3, 4, 5],
    'model__min_samples_split': [2, 5, 10]
}


#grid search with cross validation
grid_search = GridSearchCV(
    pipe, #pipeline
    param_grid, #parameters to try
    cv = 5, #five fold cross vallidation
    scoring = 'accuracy',
    n_jobs = -1 #use all cpus
)

grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.2f}")
print(f"Test score: {grid_search.score(X_test, y_test):.4f}")

Best Parameters: {'model__max_depth': 5, 'model__min_samples_split': 2, 'model__n_estimators': 100}
Best CV score: 0.81
Test score: 0.7972
